In [8]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Evaluate the model on the test set
loss, accuracy = model.evaluate(X_test_processed, y_test, verbose=0)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

# Predict probabilities on the test set
y_pred_proba = model.predict(X_test_processed)
y_pred = np.where(y_pred_proba > 0.5, 1, 0) # Convert probabilities to binary predictions

# Display classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Display confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Test Loss: 0.4700
Test Accuracy: 0.8379
70/70 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.83      0.84      1175
           1       0.81      0.85      0.83      1058

    accuracy                           0.84      2233
   macro avg       0.84      0.84      0.84      2233
weighted avg       0.84      0.84      0.84      2233


Confusion Matrix:
[[970 205]
 [157 901]]


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Separate target variable from features
X = df.drop('deposit', axis=1)
y = df['deposit']

# Convert 'deposit' target variable to numerical (0 for 'no', 1 for 'yes')
y = y.apply(lambda x: 1 if x == 'yes' else 0)

# Identify categorical and numerical features
categorical_features = X.select_dtypes(include=['object']).columns
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns

print(f"Categorical features: {list(categorical_features)}")
print(f"Numerical features: {list(numerical_features)}")

Categorical features: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']
Numerical features: ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']


Next, I will create preprocessing pipelines for both numerical and categorical features. Numerical features will be scaled using `StandardScaler`, and categorical features will be transformed using `OneHotEncoder`. These will be combined into a `ColumnTransformer`.

In [4]:
# Create preprocessing pipelines for numerical and categorical features
numerical_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

# Create a column transformer to apply different transformations to different columns
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (8929, 16)
X_test shape: (2233, 16)
y_train shape: (8929,)
y_test shape: (2233,)


Now that the data is split, I will apply the preprocessing steps to the training and test sets. This will scale numerical features and one-hot encode categorical features.

In [6]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Get the number of features after preprocessing
input_shape = X_train_processed.shape[1]

# Build the ANN model
model = Sequential([
    Dense(units=64, activation='relu', input_shape=(input_shape,)),
    Dense(units=32, activation='relu'),
    Dense(units=1, activation='sigmoid') # Sigmoid for binary classification
])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Print the model summary
model.summary()

# Train the model
history = model.fit(X_train_processed, y_train, epochs=50, batch_size=32, validation_split=0.2, verbose=1)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         3,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,441 (21.25 KB)

 Trainable params: 5,441 (21.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
224/224 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.7806 - loss: 0.4716 - val_accuracy: 0.8309 - val_loss: 0.4051
Epoch 2/50
224/224 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8361 - loss: 0.3773 - val_accuracy: 0.8399 - val_loss: 0.3854
Epoch 3/50
224/224 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8453 - loss: 0.3610 - val_accuracy: 0.8359 - val_loss: 0.3897
Epoch 4/50
224/224 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8515 - loss: 0.3497 - val_accuracy: 0.8455 - val_loss: 0.3791
Epoch 5/50
224/224 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8554 - loss: 0.3405 - val_accuracy: 0.8539 - val_loss: 0.3725
Epoch 6/50
224/224 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8571 - loss: 0.3329 - val_accuracy: 0.8533 - val_loss: 0.3692
Epoch 7/50
224/224 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8632 - loss: 0.3232 - val_accuracy: 0.8488 - val_loss: 0.3724
Epoch 8/50
224/224 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8685 - loss: 0.3186 - val_accuracy: 0.

Now that the model is trained, I will evaluate its performance on the test set and display the classification report and confusion matrix to get a detailed understanding of its accuracy, precision, recall, and F1-score.

In [9]:
# Install Keras Tuner
!pip install -q -U keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 1.6 MB/s eta 0:00:00


Now, I will define a `build_model` function that creates an ANN with tunable hyperparameters. This function will be used by the Keras Tuner to explore different model configurations.

In [10]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
import keras_tuner as kt

# Get the number of features after preprocessing
input_shape = X_train_processed.shape[1]

def build_model(hp):
    model = Sequential()
    model.add(Dense(units=hp.Int('units_1', min_value=32, max_value=128, step=32), activation='relu', input_shape=(input_shape,)))
    model.add(Dense(units=hp.Int('units_2', min_value=16, max_value=64, step=16), activation='relu'))
    model.add(Dense(units=1, activation='sigmoid'))

    # Tune the learning rate for the Adam optimizer
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])

    model.compile(optimizer=Adam(learning_rate=hp_learning_rate),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

# Instantiate the tuner
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=10,
    executions_per_trial=2,
    directory='my_dir',
    project_name='bank_deposit_ann')

# Print a summary of the search space
tuner.search_space_summary()


Search space summary
Default search space size: 3
units_1 (Int)
{'default': None, 'conditions': [], 'min_value': 32, 'max_value': 128, 'step': 32, 'sampling': 'linear'}
units_2 (Int)
{'default': None, 'conditions': [], 'min_value': 16, 'max_value': 64, 'step': 16, 'sampling': 'linear'}
learning_rate (Choice)
{'default': 0.01, 'conditions': [], 'values': [0.01, 0.001, 0.0001], 'ordered': True}


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Now, let's run the hyperparameter search to find the best model configuration. This process will train multiple models with different hyperparameter combinations.

In [11]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='keras_tuner')

# Run the hyperparameter search
tuner.search(X_train_processed, y_train, epochs=10, validation_split=0.2, verbose=1)

# Get the optimal hyperparameters
best_hps=tuner.get_best_hyperparameters(num_trials=1)[0]

print(f"The best hyperparameters are:")
print(f"- Number of units in first Dense layer: {best_hps.get('units_1')}")
print(f"- Number of units in second Dense layer: {best_hps.get('units_2')}")
print(f"- Learning rate for the optimizer: {best_hps.get('learning_rate')}")

# Build the best model
best_model = tuner.get_best_models(num_models=1)[0]

# Evaluate the best model on the test data
loss, accuracy = best_model.evaluate(X_test_processed, y_test, verbose=0)
print(f"\nBest Model Test Loss: {loss:.4f}")
print(f"Best Model Test Accuracy: {accuracy:.4f}")

# Predict probabilities on the test set with the best model
y_pred_proba_best = best_model.predict(X_test_processed)
y_pred_best = np.where(y_pred_proba_best > 0.5, 1, 0)

# Display classification report for the best model
print("\nClassification Report for Best Model:")
print(classification_report(y_test, y_pred_best))

# Display confusion matrix for the best model
print("\nConfusion Matrix for Best Model:")
print(confusion_matrix(y_test, y_pred_best))

Trial 10 Complete [00h 00m 23s]
val_accuracy: 0.8342665135860443

Best val_accuracy So Far: 0.8575027883052826
Total elapsed time: 00h 04m 13s
The best hyperparameters are:
- Number of units in first Dense layer: 128
- Number of units in second Dense layer: 48
- Learning rate for the optimizer: 0.01


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))



Best Model Test Loss: 0.3601
Best Model Test Accuracy: 0.8495
70/70 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step

Classification Report for Best Model:
              precision    recall  f1-score   support

           0       0.89      0.82      0.85      1175
           1       0.82      0.88      0.85      1058

    accuracy                           0.85      2233
   macro avg       0.85      0.85      0.85      2233
weighted avg       0.85      0.85      0.85      2233


Confusion Matrix for Best Model:
[[964 211]
 [125 933]]


In [7]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Evaluate the model on the test set
loss, accuracy = model.evaluate(X_test_processed, y_test, verbose=0)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

# Predict probabilities on the test set
y_pred_proba = model.predict(X_test_processed)
y_pred = np.where(y_pred_proba > 0.5, 1, 0) # Convert probabilities to binary predictions

# Display classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Display confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Test Loss: 0.4700
Test Accuracy: 0.8379
70/70 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.83      0.84      1175
           1       0.81      0.85      0.83      1058

    accuracy                           0.84      2233
   macro avg       0.84      0.84      0.84      2233
weighted avg       0.84      0.84      0.84      2233


Confusion Matrix:
[[970 205]
 [157 901]]


In [5]:
# Apply preprocessing to the training and test sets
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"X_train_processed shape: {X_train_processed.shape}")
print(f"X_test_processed shape: {X_test_processed.shape}")

X_train_processed shape: (8929, 51)
X_test_processed shape: (2233, 51)


In [1]:
import pandas as pd

# Try to read the file as an Excel file first, due to the .xls extension
try:
    df = pd.read_excel('/content/bank.csv.xls')
except Exception as e:
    print(f"Could not read as Excel, attempting to read as CSV: {e}")
    df = pd.read_csv('/content/bank.csv.xls')

print("Dataset loaded successfully. Here are the first 5 rows:")
display(df.head())

Could not read as Excel, attempting to read as CSV: Excel file format cannot be determined, you must specify an engine manually.
Dataset loaded successfully. Here are the first 5 rows:


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes


In [2]:
print("Dataset information:")
display(df.info())

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11162 entries, 0 to 11161
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        11162 non-null  int64 
 1   job        11162 non-null  object
 2   marital    11162 non-null  object
 3   education  11162 non-null  object
 4   default    11162 non-null  object
 5   balance    11162 non-null  int64 
 6   housing    11162 non-null  object
 7   loan       11162 non-null  object
 8   contact    11162 non-null  object
 9   day        11162 non-null  int64 
 10  month      11162 non-null  object
 11  duration   11162 non-null  int64 
 12  campaign   11162 non-null  int64 
 13  pdays      11162 non-null  int64 
 14  previous   11162 non-null  int64 
 15  poutcome   11162 non-null  object
 16  deposit    11162 non-null  object
dtypes: int64(7), object(10)
memory usage: 1.4+ MB


None